In [ ]:
import os, sys, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import euclidean_distances

In [ ]:
def random_seed(seed=1234):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

random_seed(1234)

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)


Device: cpu


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 경로 설정
ospj = os.path.join
path_workspace = '/content/drive/MyDrive/'
sys.path.append(ospj(path_workspace, 'Collected_dataset'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Define training data paths
dict_path_tr_clean = {
    #'20m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_static.csv'),
     '20m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_dyn1.csv'),
    # '20m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_dyn2.csv'),
    #'50m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_static.csv'),
     '50m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_dyn1.csv'),
    # '50m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_dyn2.csv'),
    #'70m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_static.csv'),
     '70m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_dyn1.csv'),
    # '70m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_dyn2.csv'),
}


dict_path_te_clean = {
    #'35m_stat': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_stat.csv'),
     '35m_dyn1': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_dyn1.csv'),
    # '35m_dyn2': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_dyn2.csv'),
}



# dict_path_tr_clean = {
#     '20m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_static.csv'),
#     '20m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_dyn1.csv'),
#     '20m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/20m_dyn2.csv'),
#     '50m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_static.csv'),
#     '50m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_dyn1.csv'),
#     '50m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/50m_dyn2.csv'),
#     '70m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_static.csv'),
#     '70m_dyn1': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_dyn1.csv'),
#     '70m_dyn2': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/70m_dyn2.csv'),
# }


# dict_path_te_clean = {
#     '35m_stat': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_stat.csv'),
#     '35m_dyn1': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_dyn1.csv'),
#     '35m_dyn2': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_dyn2.csv'),
# }






# dict_path_tr_clean = {
#     '35m_stat': ospj(path_workspace, r'Collected_dataset/Training & Validation/clean/35m_stat_train.csv'),
# }
# dict_path_te_clean = {
#     '35m_stat': ospj(path_workspace, r'Collected_dataset/Testing/clean/35m_stat_test.csv'),
# }





In [ ]:
selected_columns = [
    'lat','lon','alt','alt_ellipsoid',
    'vel_m_s','vel_n_m_s','vel_e_m_s','vel_d_m_s',
    'cog_rad','timestamp'
]

In [ ]:

dict_df_tr_clean, dict_df_te_clean = {}, {}
for key, p in dict_path_tr_clean.items():
    dict_df_tr_clean[key] = pd.read_csv(p, usecols=selected_columns)
    print(f'{key} ✅ Training data loaded')

for key, p in dict_path_te_clean.items():
    dict_df_te_clean[key] = pd.read_csv(p, usecols=selected_columns)
    print(f'{key} ✅ Test data loaded')


20m_dyn1 ✅ Training data loaded
50m_dyn1 ✅ Training data loaded
70m_dyn1 ✅ Training data loaded
35m_dyn1 ✅ Test data loaded


In [ ]:
def convert_timestamp_to_seconds(t):
    try:
        m, s = t.split(':')
        return int(m)*60 + float(s)
    except Exception as e:
        return np.nan

def split_by_time_gap(df, time_col='timestamp', threshold=1.0):
    d = df.copy()
    d[time_col] = d[time_col].apply(convert_timestamp_to_seconds)
    d = d.dropna(subset=[time_col]).reset_index(drop=True)
    diffs = d[time_col].diff().abs().fillna(0.0)
    idx = diffs[diffs >= threshold].index.tolist()
    idx = [0] + idx + [len(d)]
    return [ d.iloc[idx[i]:idx[i+1]].reset_index(drop=True) for i in range(len(idx)-1) ]

dict_split_tr_clean = {k: split_by_time_gap(df, threshold=1.0) for k, df in dict_df_tr_clean.items()}
dict_split_te_clean = {k: split_by_time_gap(df, threshold=1.0) for k, df in dict_df_te_clean.items()}



In [ ]:
# Train/Valid 분리 (원본 로직: 마지막 세그먼트만 검증)
dict_split_train_only, dict_split_valid_only = {}, {}
for k, segs in dict_split_tr_clean.items():
    if len(segs) > 1:
        dict_split_train_only[k] = segs[:-1]
        dict_split_valid_only[k] = [segs[-1]]
    else:
        dict_split_train_only[k] = []
        dict_split_valid_only[k] = segs


In [ ]:
def print_segment_counts(dict_split_train_only, dict_split_valid_only, dict_split_te_clean):
    print("\n📦 세그먼트 개수 요약")
    total_train, total_valid, total_test = 0, 0, 0

    print("\n[TRAIN & VALID]")
    for k in dict_split_train_only.keys():
        n_train = len(dict_split_train_only[k])
        n_valid = len(dict_split_valid_only[k])
        total_train += n_train
        total_valid += n_valid
        print(f"{k} → Train seg: {n_train}, Valid seg: {n_valid}")

    print("\n[TEST]")
    for k in dict_split_te_clean.keys():
        n_test = len(dict_split_te_clean[k])
        total_test += n_test
        print(f"{k} → Test seg: {n_test}")

    print("\n📊 전체 합계")
    print(f"Train seg total: {total_train}")
    print(f"Valid seg total: {total_valid}")
    print(f"Test seg total : {total_test}")

In [ ]:
print_segment_counts(dict_split_train_only, dict_split_valid_only, dict_split_te_clean)


📦 세그먼트 개수 요약

[TRAIN & VALID]
20m_dyn1 → Train seg: 3, Valid seg: 1
50m_dyn1 → Train seg: 3, Valid seg: 1
70m_dyn1 → Train seg: 4, Valid seg: 1

[TEST]
35m_dyn1 → Test seg: 4

📊 전체 합계
Train seg total: 10
Valid seg total: 3
Test seg total : 4


In [ ]:
# ======================================
# 4) 파생 피처 + 윈도잉 (원본 로직 유지)
# ======================================
def add_delta_coordinates(dict_split):
    for segs in dict_split.values():
        for df in segs:
            df['D_lat'] = [0.0] + (df['lat'].diff().fillna(0).tolist())[1:]
            df['D_lon'] = [0.0] + (df['lon'].diff().fillna(0).tolist())[1:]
            df['D_alt'] = [0.0] + (df['alt'].diff().fillna(0).tolist())[1:]

def add_euc_distance(dict_split):
    for segs in dict_split.values():
        for df in segs:
            coords = df[['lat','lon','alt']].to_numpy()
            distances = [0.0]
            for i in range(1, len(coords)):
                distances.append(euclidean_distances([coords[i-1]],[coords[i]])[0][0])
            df['d_pos'] = distances

def add_time_gap(dict_split):
    for segs in dict_split.values():
        for df in segs:
            df['time_gap'] = [0.0] + (df['timestamp'].diff().fillna(0).tolist())[1:]

for d in [dict_split_train_only, dict_split_valid_only, dict_split_te_clean]:
    add_delta_coordinates(d); add_euc_distance(d); add_time_gap(d)




In [ ]:
window_size = 5  # T
input_features  = ['vel_m_s','vel_n_m_s','vel_e_m_s','cog_rad','D_lat','D_lon','d_pos','time_gap']
target_features = ['D_lon']
#target_features = ['D_lat','D_lon']
# target features = d_lat 만




In [ ]:
def create_sequence_label_tensors(dict_split, window_size):
    X_all, y_all = [], []
    for segs in dict_split.values():
        for df in segs:
            if len(df) <= window_size: continue
            data   = df[input_features].to_numpy()
            labels = df[target_features].to_numpy()
            for i in range(len(df) - window_size):
                X_all.append(data[i : i + window_size])
                y_all.append(labels[i + window_size])
    return np.array(X_all), np.array(y_all)

X_train, y_train = create_sequence_label_tensors(dict_split_train_only, window_size)
X_valid, y_valid = create_sequence_label_tensors(dict_split_valid_only, window_size)
X_test,  y_test  = create_sequence_label_tensors(dict_split_te_clean,  window_size)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_valid: {X_valid.shape}, y_valid: {y_valid.shape}")
print(f"X_test : {X_test.shape},  y_test : {y_test.shape}")

X_train: (4394, 5, 8), y_train: (4394, 1)
X_valid: (1347, 5, 8), y_valid: (1347, 1)
X_test : (2100, 5, 8),  y_test : (2100, 1)


In [ ]:
# ======================================
# 5) DataLoader (AE / 지도학습)
# ======================================
def make_loader_ae(X, batch_size=64, shuffle=True):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    return DataLoader(TensorDataset(X_tensor, X_tensor), batch_size=batch_size, shuffle=shuffle)

def make_loader_sup(X, y, batch_size=64, shuffle=True):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=shuffle)

train_loader_ae = make_loader_ae(X_train, batch_size=64, shuffle=True)
valid_loader_ae = make_loader_ae(X_valid, batch_size=64, shuffle=False)
train_loader_sup = make_loader_sup(X_train, y_train, batch_size=64, shuffle=True)
valid_loader_sup = make_loader_sup(X_valid, y_valid, batch_size=64, shuffle=False)
test_loader_sup  = make_loader_sup(X_test,  y_test,  batch_size=64, shuffle=False)

input_dim = X_train.shape[-1]
seq_len   = X_train.shape[1]


In [ ]:
import torch.nn as nn

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.2, seq_len=5):
        super().__init__()
        self.seq_len = seq_len
        self.input_dim = input_dim
        self.encoder = nn.LSTM(
            input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers>1 else 0.0
        )
        self.decoder = nn.LSTM(
            input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers>1 else 0.0
        )
        self.proj = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        _, (h, c) = self.encoder(x)
        B = x.size(0)
        dec_in = torch.zeros(B, self.seq_len, self.input_dim, device=x.device)
        dec_out, _ = self.decoder(dec_in, (h, c))
        return self.proj(dec_out)

def train_ae_with_early_stop(
    input_dim, seq_len, train_loader, valid_loader,
    n_epochs=1000, lr=1e-3, hidden_dim=128, num_layers=2, dropout=0.2,
    patience=40, device=device
):
    model = LSTMAutoencoder(input_dim, hidden_dim, num_layers, dropout, seq_len).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    crit = nn.MSELoss()

    best_mse = float('inf')
    best_state = None
    best_epoch = -1
    no_imp = 0

    for ep in range(1, n_epochs+1):
        # Train
        model.train(); tr=[]
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            xhat = model(xb)
            loss = crit(xhat, yb)
            loss.backward()
            opt.step()
            tr.append(loss.item())
        tr_mse = float(np.mean(tr)) if tr else np.nan

        # Valid
        model.eval(); va=[]
        with torch.no_grad():
            for xb, yb in valid_loader:
                xb, yb = xb.to(device), yb.to(device)
                va.append(crit(model(xb), yb).item())
        va_mse = float(np.mean(va)) if va else np.nan

        if (ep % 5 == 0) or (va_mse < best_mse):
            print(f"[AE] Epoch {ep}/{n_epochs} | Train MSE {tr_mse:.6f} | Valid MSE {va_mse:.6f} | Best {best_mse:.6f} (ep {best_epoch})")

        if va_mse < best_mse:
            best_mse = va_mse
            best_state = {k:v.cpu() for k,v in model.state_dict().items()}
            best_epoch = ep
            no_imp = 0
            print("[AE] 🎯 New Best Valid MSE!")
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f"[AE] ⏹️ Early stopping at epoch {ep}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"[AE] ✅ Finished. Best Valid MSE: {best_mse:.6f} at epoch {best_epoch}")
    return model, best_mse, best_epoch

In [ ]:
lstm_hidden_dim = 128
lstm_layers     = 2
dropout         = 0.2
ae_model, best_mse, best_ep = train_ae_with_early_stop(
    input_dim=input_dim, seq_len=seq_len,
    train_loader=train_loader_ae, valid_loader=valid_loader_ae,
    n_epochs=1000, lr=0.0001,
    hidden_dim=lstm_hidden_dim, num_layers=lstm_layers, dropout=dropout,
    patience=200, device=device
)

[AE] Epoch 1/1000 | Train MSE 1302.885786 | Valid MSE 1217.276083 | Best inf (ep -1)
[AE] 🎯 New Best Valid MSE!
[AE] Epoch 2/1000 | Train MSE 1228.687619 | Valid MSE 1133.234079 | Best 1217.276083 (ep 1)
[AE] 🎯 New Best Valid MSE!
[AE] Epoch 3/1000 | Train MSE 1174.810332 | Valid MSE 1102.847302 | Best 1133.234079 (ep 2)
[AE] 🎯 New Best Valid MSE!
[AE] Epoch 4/1000 | Train MSE 1148.782853 | Valid MSE 1083.514334 | Best 1102.847302 (ep 3)
[AE] 🎯 New Best Valid MSE!
[AE] Epoch 5/1000 | Train MSE 1131.166593 | Valid MSE 1067.591287 | Best 1083.514334 (ep 4)
[AE] 🎯 New Best Valid MSE!
[AE] Epoch 6/1000 | Train MSE 1114.817466 | Valid MSE 1053.422444 | Best 1067.591287 (ep 5)
[AE] 🎯 New Best Valid MSE!
[AE] Epoch 7/1000 | Train MSE 1098.931513 | Valid MSE 1040.034845 | Best 1053.422444 (ep 6)
[AE] 🎯 New Best Valid MSE!
[AE] Epoch 8/1000 | Train MSE 1084.869706 | Valid MSE 1027.347839 | Best 1040.034845 (ep 7)
[AE] 🎯 New Best Valid MSE!
[AE] Epoch 9/1000 | Train MSE 1070.842650 | Valid MSE 1

In [ ]:
# ======================================
# 7) AE 인코더 재사용 LSTM+MLP 예측 모델
# ======================================
class AEEncoderRegressor(nn.Module):
    def __init__(self, ae_model, head_lstm_hidden=128, head_lstm_layers=1,
                 mlp_layers=2, output_dim=2, dropout=0.2, freeze_encoder=False):
        super().__init__()
        # AE encoder와 동일 사양의 LSTM 생성 후 state_dict 복사
        self.encoder = nn.LSTM(
            input_size=ae_model.encoder.input_size,
            hidden_size=ae_model.encoder.hidden_size,
            num_layers=ae_model.encoder.num_layers,
            batch_first=True,
            dropout=dropout if ae_model.encoder.num_layers>1 else 0.0
        )
        self.encoder.load_state_dict(ae_model.encoder.state_dict())
        if freeze_encoder:
            for p in self.encoder.parameters(): p.requires_grad=False

        enc_outdim = ae_model.encoder.hidden_size
        self.head_lstm = nn.LSTM(
            input_size=enc_outdim, hidden_size=head_lstm_hidden,
            num_layers=head_lstm_layers, batch_first=True,
            dropout=dropout if head_lstm_layers>1 else 0.0
        )

        mlp = []
        in_dim = head_lstm_hidden
        for _ in range(mlp_layers):
            out_dim = max(in_dim//2, 8)
            mlp += [nn.Linear(in_dim, out_dim), nn.ReLU()]
            in_dim = out_dim
        mlp += [nn.Linear(in_dim, output_dim)]
        self.mlp = nn.Sequential(*mlp)

    def forward(self, x):
        enc_out,_  = self.encoder(x)      # (B, T, H_enc)
        head_out,_ = self.head_lstm(enc_out) # (B, T, H_head)
        last = head_out[:, -1, :]         # (B, H_head)
        return self.mlp(last)             # (B, 2)



In [ ]:
reg_model = AEEncoderRegressor(
    ae_model=ae_model,
    head_lstm_hidden=128,
    head_lstm_layers=2,
    mlp_layers=3, # 3,4,5 등등
    output_dim=1,
    dropout=dropout,
    freeze_encoder=False  # 필요 시 True로 바꾸면 인코더 동결
)
#mse lr 을보니까 range 가 구간이있는데 몰려있는곳,autoincoder는 구간이 짧은네 분표되어있어 전반적으로는 auto인코더가 좋다고 그걸 summary 스탯으로 엑셀로 loss 받아놓기.
# test 파일로.

In [ ]:
# ======================================
# 8) 원하는 로그 포맷의 트레이너 + 학습 실행
# ======================================
import numpy as np

def compute_all_metrics(preds, targets):
    pn = preds.detach().cpu().numpy()
    tn = targets.detach().cpu().numpy()
    mse  = float(np.mean((pn - tn)**2))
    rmse = float(np.sqrt(mse))
    mae  = float(np.mean(np.abs(pn - tn)))
    return {"MSE": mse, "RMSE": rmse, "MAE": mae}

def train_with_valid_early_stop(
    model,                    # 이미 생성된 모델 인스턴스 (reg_model)
    train_loader, valid_loader,
    n_epochs=300, lr=1e-3, weight_decay=1e-5,
    patience=30, log_every=5, device=device
):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    train_mse_curve, valid_mse_curve = [], []
    best_mse = float('inf')
    best_state = None
    best_epoch = 0
    no_improve = 0

    for epoch in range(n_epochs):
        # === Train ===
        model.train()
        all_preds, all_targets = [], []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            all_preds.append(preds.detach().cpu())
            all_targets.append(y_batch.detach().cpu())
        train_mse = compute_all_metrics(torch.cat(all_preds), torch.cat(all_targets))['MSE']
        train_mse_curve.append(train_mse)

        # === Valid ===
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for X_batch, y_batch in valid_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                preds = model(X_batch)
                val_preds.append(preds.detach().cpu())
                val_targets.append(y_batch.detach().cpu())
        valid_mse = compute_all_metrics(torch.cat(val_preds), torch.cat(val_targets))['MSE']
        valid_mse_curve.append(valid_mse)

        # 로그 (요청한 포맷 그대로)
        if (epoch + 1) % log_every == 0 or valid_mse < best_mse:
            print(f"Epoch {epoch+1}/{n_epochs} - "
                  f"Train MSE: {train_mse:.6f} | Valid MSE: {valid_mse:.6f} | "
                  f"Best Valid MSE: {best_mse:.6f} (Epoch {best_epoch+1})")

        # Early stopping
        if valid_mse < best_mse:
            best_mse = valid_mse
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            best_epoch = epoch
            no_improve = 0
            print("🎯 New Best Valid MSE!")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"⏹️ Early stopping triggered at epoch {epoch+1}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"✅ Finished. Best Valid MSE: {best_mse:.6f} at Epoch {best_epoch+1}")

    return model, train_mse_curve, valid_mse_curve, best_mse, best_epoch



In [ ]:
reg_model, tr_curve, va_curve, best_mse, best_ep = train_with_valid_early_stop(
    model=reg_model,
    train_loader=train_loader_sup,
    valid_loader=valid_loader_sup,
    n_epochs=1000, lr=0.000001, weight_decay=0.2,
    patience=200, log_every=5, device=device
)

Epoch 1/1000 - Train MSE: 2015.142456 | Valid MSE: 1776.600464 | Best Valid MSE: inf (Epoch 1)
🎯 New Best Valid MSE!
Epoch 2/1000 - Train MSE: 2015.134521 | Valid MSE: 1776.591553 | Best Valid MSE: 1776.600464 (Epoch 1)
🎯 New Best Valid MSE!
Epoch 3/1000 - Train MSE: 2015.124756 | Valid MSE: 1776.582397 | Best Valid MSE: 1776.591553 (Epoch 2)
🎯 New Best Valid MSE!
Epoch 4/1000 - Train MSE: 2015.113281 | Valid MSE: 1776.573486 | Best Valid MSE: 1776.582397 (Epoch 3)
🎯 New Best Valid MSE!
Epoch 5/1000 - Train MSE: 2015.105103 | Valid MSE: 1776.564209 | Best Valid MSE: 1776.573486 (Epoch 4)
🎯 New Best Valid MSE!
Epoch 6/1000 - Train MSE: 2015.094849 | Valid MSE: 1776.554932 | Best Valid MSE: 1776.564209 (Epoch 5)
🎯 New Best Valid MSE!
Epoch 7/1000 - Train MSE: 2015.085327 | Valid MSE: 1776.545654 | Best Valid MSE: 1776.554932 (Epoch 6)
🎯 New Best Valid MSE!
Epoch 8/1000 - Train MSE: 2015.075073 | Valid MSE: 1776.536011 | Best Valid MSE: 1776.545654 (Epoch 7)
🎯 New Best Valid MSE!
Epoch 9/

In [ ]:
# # ======================================
# # 9) 테스트 평가 + 10개 출력 & 저장
# # ======================================
# reg_model.eval()
# preds, trues = [], []
# with torch.no_grad():
#     for xb, yb in test_loader_sup:
#         xb = xb.to(device)
#         preds.append(reg_model(xb).cpu().numpy())
#         trues.append(yb.numpy())
# preds = np.vstack(preds)
# trues = np.vstack(trues)

# mse = mean_squared_error(trues, preds)
# rmse = float(np.sqrt(mse))
# print(f"[TEST] MSE={mse:.8f} | RMSE={rmse:.8f}")

# # 앞 10개 비교
# n_show = min(10, preds.shape[0])
# df_show = pd.DataFrame({
#     "true_D_lat": trues[:n_show, 0],
#     "pred_D_lat": preds[:n_show, 0],
#     "true_D_lon": trues[:n_show, 1],
#     "pred_D_lon": preds[:n_show, 1],
# })
# print("\n🔎 First 10 predictions vs truth (Δlat, Δlon):")
# print(df_show.round(8))

# # 전체 저장
# save_csv = "/content/drive/MyDrive/test_next_delta_latlon_pred_from_AEencoder.csv"
# pd.DataFrame({
#     "true_D_lat": trues[:, 0], "pred_D_lat": preds[:, 0],
#     "true_D_lon": trues[:, 1], "pred_D_lon": preds[:, 1]
# }).to_csv(save_csv, index=False)
# print(f"✅ Saved: {save_csv}")


D_lon 만

In [ ]:
# ======================================
# 9) 테스트 평가 + 10개 출력 (D_lon 전용) — CSV 저장 제거/정리
# ======================================
reg_model.eval()
preds_list, trues_list = [], []
with torch.no_grad():
    for xb, yb in test_loader_sup:
        xb = xb.to(device)
        preds_list.append(reg_model(xb).squeeze(-1).cpu())  # (B,)
        trues_list.append(yb.squeeze(-1).cpu())             # (B,)

# 하나로 합치고 numpy로 변환
preds = torch.cat(preds_list).numpy()
trues = torch.cat(trues_list).numpy()

# 메트릭
errors = preds - trues
mse  = float(np.mean(errors**2))
rmse = float(np.sqrt(mse))
mae  = float(np.mean(np.abs(errors)))
print(f"[TEST] (D_lon) MSE={mse:.8f} | RMSE={rmse:.8f} | MAE={mae:.8f}")

# 앞 10개 비교 출력
n_show = min(10, preds.shape[0])
print("\n🔎 First 10 predictions vs truth (Δlat):")
print(pd.DataFrame({
    "true_D_lon": trues[:n_show],
    "pred_D_lon": preds[:n_show],
    "error":      errors[:n_show],
}).round(8))

# ===== Summary Statistics for D_lon =====
df_res = pd.DataFrame({
    "true_D_lon":   trues,
    "pred_D_lon":   preds,
    "error":        errors,
    "abs_error":    np.abs(errors),
    "squared_error": errors**2
})
print("\n📊 Summary Statistics (D_lon):")
print(df_res.describe().round(6))


[TEST] (D_lon) MSE=112.15049744 | RMSE=10.59011319 | MAE=7.75546122

🔎 First 10 predictions vs truth (Δlat):
   true_D_lon  pred_D_lon      error
0       -14.0   -7.215396   6.784604
1       -14.0   -4.869459   9.130541
2       -17.0   -6.456611  10.543389
3       -16.0   -7.907961   8.092038
4       -15.0   -8.360564   6.639435
5       -10.0  -10.102258  -0.102258
6       -13.0  -11.021634   1.978366
7       -12.0  -11.271806   0.728194
8        -7.0   -9.886579  -2.886579
9        -7.0   -8.845584  -1.845584

📊 Summary Statistics (D_lon):
        true_D_lon   pred_D_lon        error    abs_error  squared_error
count  2100.000000  2100.000000  2100.000000  2100.000000    2100.000000
mean      0.914286     2.531928     1.617642     7.755461     112.150497
std      22.791069    14.552610    10.468329     7.213050     173.495697
min     -51.000000   -15.813705   -24.467789     0.001835       0.000003
25%      -5.000000    -4.867632    -5.397202     1.875941       3.519155
50%       0.000